<a href="https://colab.research.google.com/github/tipismaca/Trabajo-Bebidas-del-Maipo-SA/blob/main/Codigo/Entrega2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Caso de Estudio: Bebidas del Maipo S.A. — Entrega 2
**Autoras:** Maytte Almuna, Macarena Campos, Darlyn Jerez y Valentina Roa


## 1. Configuración inicial
Clonación del repositorio y librerías

In [ ]:
# Clona el repositorio para tener acceso a los datos
!rm -rf Trabajo-Bebidas-del-Maipo-SA
!git clone https://github.com/tipismaca/Trabajo-Bebidas-del-Maipo-SA.git

Cloning into 'Trabajo-Bebidas-del-Maipo-SA'...
remote: Enumerating objects: 514, done.
remote: Counting objects: 100% (160/160), done.
remote: Compressing objects: 100% (140/140), done.
remote: Total 514 (delta 68), reused 60 (delta 16), pack-reused 354 (from 1)
Receiving objects: 100% (514/514), 41.67 MiB | 20.52 MiB/s, done.
Resolving deltas: 100% (261/261), done.


In [ ]:
%pip install -q "pulp>=2.7,<3"   # PuLP 2.x trae CBC incluido

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pulp
from IPython.display import display


## 2. Carga de datos

In [ ]:
ruta_base = "Trabajo-Bebidas-del-Maipo-SA/Datos_crudos/"
ruta_resultados = "Trabajo-Bebidas-del-Maipo-SA/Resultados/"



---
## Parte A1. Balance de inventario por sku y semana.


In [ ]:
# =====================================================================
# 2.1 DATOS CRUDOS DEL HORIZONTE TÁCTICO
# =====================================================================
maestro      = pd.read_csv(ruta_base + "maestro_sku.csv")
inv_inicial  = pd.read_csv(ruta_base + "inventario_inicial.csv")
capacidad    = pd.read_csv(ruta_base + "capacidad_horizonte.csv")
setups       = pd.read_csv(ruta_base + "setups_cip.csv")
compromisos  = pd.read_csv(ruta_base + "compromisos_semana_focal.csv")

# =====================================================================
# 2.2 TABLAS DE TRASPASO DE LA ENTREGA 1 (Parte C)
# =====================================================================
tabla_c1 = pd.read_csv(ruta_resultados + "merma_por_linea.csv")
tabla_c2 = pd.read_csv(ruta_resultados + "sobrellenado_por_formato.csv")
tabla_c3 = pd.read_csv(ruta_resultados + "pronostico_sku_horizonte.csv")
tabla_c4 = pd.real_csv(ruta_resultados + "sigma_error_sku_horizonte.csv")

# C.1: ρ_ℓ por línea
rho = dict(zip(tabla_c1["Línea"], tabla_c1["ρ (proporción)"].astype(float)))

# C.2: litros de jarabe adicionales por caja, por formato (se usa en A3)
jarabe_extra_L_caja = dict(zip(tabla_c2["Formato (L)"].astype(float),
                               tabla_c2["Jarabe adicional (L/caja)"].astype(float)))

# C.3 y C.4: matriz SKU × (h=1..h=13) a formato largo (sku, h, valor)
def matriz_a_largo(tabla, nombre_valor):
    largo = tabla.melt(id_vars=["Familia", "SKU"], var_name="h", value_name=nombre_valor)
    largo["h"] = largo["h"].str.replace("h=", "", regex=False).astype(int)
    largo[nombre_valor] = largo[nombre_valor].astype(float)
    return largo.rename(columns={"SKU": "sku", "Familia": "familia"})